### Clustering

- I need to do some clustering algos and see which does best
- Select which columns to use


What am I hoping clustering will do?
- Make more "like" groupings of school districts based on certain demographic data. 
- I believe the clusters will still have normal distribution curves, and linear relationships.

What columns are allowed in clustering -- For this one I want to make two selection types, one with ethnic data and one without as some grants or other funding does not like considering that information when benchmarking
- Eco-dis %
- student body size
- average teacher pay
- Ethnic homogeneity with students and staff
- Staff size


What makes up a row?
- a row will be made up of one school with all of its given metrics
- I want to only select schools who's metrics have not drastically changed in the past 3 years. What is why I have the historical data as I believe if a school has hd drastic changes they will be an outlier and throw off the model. for the school years data I will want to use the 2024 numbers as I believe any miss calculations have been adjusted by then (not sure if they even upload corrections but don't feel like looking as of now).
- I will need to normalize all the numbers down to aq 0 to 1 scale so large differences aren't causing issues.

I would also like to run an algo to see how strongly any one variable seems to be in relation to the overall outcomes

Right now, my data probably looks like a few loosely clouds semi clustered and overlapping because nothing is normalized and teacher pay of $40,000 and eco-dis count of 270 out of 3000 students are working together to make the a shape but are working on completely different scales, so I need an approach that with a very loss clustering algo until I clean further.

In [ ]:
# Importing libraries
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns


In [ ]:
# loading in data sets
student_df = pd.read_csv(r"Clean Datasets\student_analysis_df.csv", dtype={'YearDistrict ID': str, 'DISTRICT': str, 'Year': str})
staff_df = pd.read_csv(r"Clean Datasets\staff_analysis_df.csv", dtype={'YearDistrict ID': str, 'DISTRICT': str, 'Year': str})


## Exploratory Clustering

- create two groupings og teachers
- Z-Score normalize
- 

- Stretch goal: Principal Components Analysis (PAC) on eco-dis, homeless


# Note something to consider is grouping teachers experience level

In [ ]:
# Defining columns to keep

Id_col = ['YearDistrict ID', 'DISTRICT', 'Year']


student_col_to_keep = ['YearDistrict ID', 'DISTRICT', 'Year', 'All Students Count', 'Special Ed Count',
                        'Bilingual/ESL Count', 'Career & Technical Education Count',
                        'Gifted & Talented Count', 'EB/EL Count', 'Econ Disadv Count',
                        'Non-Educationally Disadv Count', 'At Risk Count',
                        'American Indian Count', 'Asian Count', 'Pacific Islander Count',
                        'Two or More Races Count', 'African American Count', 'Hispanic Count',
                        'White Count', 'Male Count', 'Female Count',
                        'Section 504 Count', 'Title I Count', 'Homeless Count',
                        'Immigrant Count', 'Migrant Count',
                        'Total Students with Disabilities Count',
                        'Intellectual Disabilities Count', 'Physical Disabilities Count',
                        'Behavioral Disabilities Count', 'District Name', 'County ID',
                        'County Name', 'Region', 'Charter Flag', 'DFLALTED']

staff_col_to_keep = [   'YearDistrict ID', 'Year', 'DISTRICT',       
                        'Teacher Total Full Time Equiv Count',
                        'Support Total Full Time Equiv Count',
                        'School Admin Total Full Time Equiv Count',
                        'Central Admin Total Full Time Equiv Count',
                        'Educ Aide Total Full Time Equiv Count',
                        'Teacher Total Base Salary Total', 'Support Total Base Salary Total',
                        'School Admin Total Base Salary Total',
                        'Teacher Beginning Full Time Equiv Count',
                        'Teacher 1-5 Years Full Time Equiv Count',
                        'Teacher 6-10 Years Full Time Equiv Count',
                        'Teacher 11-20 Years Full Time Equiv Count',
                        'Teacher 21-30 Years Full Time Equiv Count',
                        'Teacher > 30 Years Full Time Equiv Count',
                        'Teacher Beginning Base Salary Total',
                        'Teacher 1-5 Years Base Salary Total',
                        'Teacher 6-10 Years Base Salary Total',
                        'Teacher 11-20 Years Base Salary Total',
                        'Teacher 21-30 Years Base Salary Total',
                        'Teacher > 30 Years Base Salary Total',
                        'Teacher No Degree Full Time Equiv Count',
                        'Teacher BA Degree Full Time Equiv Count',
                        'Teacher MS Degree Full Time Equiv Count',
                        'Teacher PH Degree Full Time Equiv Count',
                        'Teacher American Indian Full Time Equiv Count',
                        'Teacher Pacific Islander Full Time Equiv Count',
                        'Teacher Asian Full Time Equiv Count',
                        'Teacher African American Full Time Equiv Count',
                        'Teacher Hispanic Full Time Equiv Count',
                        'Teacher White Full Time Equiv Count',
                        'Teacher Two or more races Full Time Equiv Count',
                        'Teacher Male Full Time Equiv Count',
                        'Teacher Female Full Time Equiv Count',
                        'Teacher Regular Program Full Time Equiv Count',
                        'Teacher Career & Technical Prgms Full Time Equiv Count',
                        'Teacher Bilingual Program Full Time Equiv Count',
                        'Teacher Compensatory Program Full Time Equiv Count',
                        'Teacher Gifted & Talented Program Full Time Equiv Count',
                        'Teacher Special Education Full Time Equiv Count',
                        'Teacher Other Full Time Equiv Count', 'Teacher Turnover Numerator',
                        'Teacher Turnover Denominator', 'Principal Experience Total',
                        'Principal Tenure Total', 'Assistant Principal Experience Total',
                        'Assistant Principal Tenure Total',
                        'Teacher Incentive Allotment Master Head Count',
                        'Teacher Incentive Allotment Exemplary Head Count',
                        'Teacher Incentive Allotment Recognized Head Count']

In [ ]:
# FILTER: filtering data frames down to columns defined to keep
student_df = student_df[student_col_to_keep]
staff_df = staff_df[staff_col_to_keep]


### Beta-Binomial Model - Smoothed Rate Function (Explore shrinkage estimation and  smoothing tequinices)

Beta-Binomial shrinkage

Notes: A discovery happened as I was exploring my data sets post z-score calculations, and that was that some districts were in the range of (10, 9), but when I just open the data set, it looks like most of the numbers are in the range (1, -1). While I did not clearly know why, from what I have learned so far about statistical modeling, I could tell such huge ranges did not make sense, so I looked at the affected columns.

When I started exploring the affected columns, I noticed that often the cell that I calculated off of for example, the number of teachers with 30+ years of experience it may have been zero because it was a small district. Initially, I wanted to keep the data set whole. But I knew I could not just make the values NaN, as that would affect downstream statistical values by either creating them inaccurately or removing them from the data set. I wanted to try and keep as many rows or values as possible, because otherwise, I'd have to remove them from multiple years, and that would eventually shrink my data set to what I believe will be unusable because I'm only working within a thousand or so data points.

In school districts with small populations, even a single data point can create an extreme outlier. For example, in a district of only five teachers, one teacher constitutes 20% of the total; that would likely create a huge statistical anomaly in my calculations down the line. To prevent these small-population "flukes" from distorting our model, we apply a smoothing technique. This approach provides "statistical help" to smaller districts by pulling their rates toward the state average, ensuring that their limited data size doesn't create misleading extremes that skew the overall analysis.
 

In [ ]:
# FUNCTION: Beta binomial shrinkage/smoothing

def bbm_smoothing(df, trials, successes, fake_trials=100):
    """
    This is a beta-binomial smoothing model, which is a more sophisticated way of determining a rate. 
    In situations with small sample sizes like a school district with only 10 teachers the small headcount 
    means theres a lower probability of seeing someone with 30-plus years of experience. Conversely, if a 
    small district happens to have 3 teachers in that category, it creates an extreme percentage that is 
    statistically unlikely but possible. My goal is to support these smaller districts by smoothing those values, ensuring 
    they aren't penalized by extreme outliers when we group or benchmark them later on.

    df: DataFrame
    Trial: In this case, the labor market of teachers in a given year.
    Success: Teachers with 30+ years of experience.
    fake_trials: How much assistance we give schools when calculating the smoothed rate.
    """
    
    # If we are just smoothing one column we need to make it a list item for the function to handel it
    if isinstance(successes, str):
        success = [successes]

    # Compute the smoothed rate for each column we are looking at.
    for success in successes:
        df[success].fillna(0)
        n_trials = df.groupby('Year')[trials].transform('sum')      # Total number of trials
        k_successes = df.groupby('Year')[success].transform('sum')  # Total number of events that meet our criteria
        probability = k_successes/n_trials                          # Rate of successes to trials
        fake_successes = probability*fake_trials                    # Alpha number of added fake successes

        df[success+"_bbm_smoothed"] = (df[success] + fake_successes)/(df[trials].fillna(0) + fake_trials)

    
    return df


In [ ]:
# FUNCTION: Euclidean Distance
def calculate_euclidean_distance(df, columns, grouping_col='Year', central_tendency='mean', output_name=None):
    """
    Compute the euclidean distance of various features to each other
    """
    if isinstance(columns, str):
        columns = [columns]

    df = df.copy()

    def compute_reference_vector(group):
        """
        This function finds the reference vector we use to see how distance our comparison point is to the reference.

        In my example i want to find the mean or average ethnicity of a student or staff from 
        """
        
        if central_tendency == 'median':
            return group[columns].median()
        else:
            return group[columns].mean()

    def euclidean_distance_from_reference(row, reference_vector):
        """
        looking at the a given row we collect all of our vectors we want to look at and compute the distance from the reference
         
        euclidean distance = sqrt(x)
        """

        comparison_vector = row[columns].values.astype(float)
        diff = comparison_vector - reference_vector.values.astype(float)
        return np.sqrt(np.sum(diff ** 2))
    
    distance = []

    for grouping_value, group in df.groupby(grouping_col):
        ref_vector = compute_reference_vector(group)

        group_distance = group.apply(
            lambda row: euclidean_distance_from_reference(row, ref_vector),
            axis=1
        )

        distance.append(group_distance)

        output_col_name = f"{output_name}_euclidean_distance"

        df[output_col_name] = pd.concat(distance).reindex(df.index)
    
    return df

In [ ]:
# Function Herfindahl-Hirschman Index (HHI) - Measures how dominant any one group in in a market
def calculate_hhi(df, columns, output_col_name=None):
    """
    computes the Herfindahl-Hirschman Index (HHI) of a column, used only on columns who are with in a grouping whos sum equals 1
    for example ethnicity distribution. where the distribution is part of the whole "market"

    HHI formula is the sum of all the squared groups within the "market"

    HHI=0.3^2 + 0.1^2 + 0.6^2
    
    0.3, 0.1, 0.6 = the market share percentage of a group (for this example students and teachers ethnicity)
    """
    if isinstance(columns, str):
        columns = [columns]

    df = df.copy()

    values = df[columns]

    hhi = (values ** 2).sum(axis=1)
    output_name = f"{output_col_name}_hhi"
    df[output_name] = hhi

    return df

In [ ]:
# FUNCTION: Calculating z score
def zscore(df: pd.DataFrame):
    """
    Calculates the z score of a column grouped by year
    df: DataFrame
    """
    for col in df.columns:
        if "smoothed" in col:
            z_col = col+"_ZScore"
            df[z_col] = (
                df[col] -
                df.groupby('Year')[col].transform('mean')
            ) / df.groupby('Year')[col].transform('std')

    return df

In [ ]:
# Select Student columns to bbm smooth

student_bbm_col = [ 'Career & Technical Education Count', 'Gifted & Talented Count',
                    'EB/EL Count','Econ Disadv Count', 'Non-Educationally Disadv Count',
                    'At Risk Count', 'Male Count', 'Female Count','Homeless Count', 'Immigrant Count', 'Migrant Count',
                    'American Indian Count', 'Asian Count','Pacific Islander Count', 'Two or More Races Count',
                    'African American Count', 'Hispanic Count', 'White Count']

In [ ]:
student_df.columns

In [ ]:
# Defining staff numerators

# Teacher Experience Levels
teacher_ep_num  =[  'Teacher Beginning Full Time Equiv Count',
                    'Teacher 1-5 Years Full Time Equiv Count',
                    'Teacher 6-10 Years Full Time Equiv Count',
                    'Teacher 11-20 Years Full Time Equiv Count',
                    'Teacher 21-30 Years Full Time Equiv Count',
                    'Teacher > 30 Years Full Time Equiv Count']

# Teacher pay ranges
teacher_pay_num = ['Teacher Beginning Base Salary Total',
                        'Teacher 1-5 Years Base Salary Total',
                        'Teacher 6-10 Years Base Salary Total',
                        'Teacher 11-20 Years Base Salary Total',
                        'Teacher 21-30 Years Base Salary Total',
                        'Teacher > 30 Years Base Salary Total']

# Teacher Education Levels
teacher_edu_num = ['Teacher No Degree Full Time Equiv Count',
                        'Teacher BA Degree Full Time Equiv Count',
                        'Teacher MS Degree Full Time Equiv Count',
                        'Teacher PH Degree Full Time Equiv Count',]

teacher_eth_num = ['Teacher American Indian Full Time Equiv Count',
                        'Teacher Pacific Islander Full Time Equiv Count',
                        'Teacher Asian Full Time Equiv Count',
                        'Teacher African American Full Time Equiv Count',
                        'Teacher Hispanic Full Time Equiv Count',
                        'Teacher White Full Time Equiv Count',
                        'Teacher Two or more races Full Time Equiv Count',]

In [ ]:
# Smoothing Student
student_df_1 = bbm_smoothing(student_df,'All Students Count', student_bbm_col) 

#staff_df_1.fillna(0, inplace=True)
staff_df_calculated_col = bbm_smoothing(staff_df,'Teacher Total Full Time Equiv Count', teacher_ep_num)
#staff_df_2 = bbm_smoothing(staff_df_2, 'Teacher Total Full Time Equiv Count', teacher_edu_numerator,)
#staff_df_2 = bbm_smoothing(staff_df_2, 'Teacher Total Full Time Equiv Count', teacher_eth_numerator,)
# staff_df_2 = bbm_smoothing(staff_df_2, 'Teacher Total Base Salary Total', 'Teacher Total Full Time Equiv Count')
# staff_df_2 = bbm_smoothing(staff_df_2, teacher_pay_numerator, 'Teacher Total Base Salary Total')


In [ ]:
student_df_1.columns

In [ ]:
bbm_student_col = ['American Indian Count_bbm_smoothed', 'Asian Count_bbm_smoothed',
       'Pacific Islander Count_bbm_smoothed',
       'Two or More Races Count_bbm_smoothed',
       'African American Count_bbm_smoothed', 'Hispanic Count_bbm_smoothed',
       'White Count_bbm_smoothed']

In [ ]:
student_df_1 = calculate_hhi(student_df_1, bbm_student_col, "student_ethnicity")
student_df_1 = calculate_euclidean_distance(student_df_1, bbm_student_col, output_name='student_ethnicity')

In [ ]:
student_24 = student_df_1[student_df_1['Year']=="2024"]

In [ ]:
student_24.columns

student_24['groups'] = pd.qcut(student_24['All Students Count'], q=4).astype(str)


### Exploring data shape

In [ ]:
sns.scatterplot(
    data=student_24,
    x='student_ethnicity_hhi',
    y='student_ethnicity_euclidean_distance',
    hue=pd.qcut(student_24['All Students Count'], q=6)
)

# High HHI = Homogeneous; Low HHI = Diverse

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

groups = pd.qcut(student_24['All Students Count'], q=6)

scatter = ax.scatter(
    student_24['student_ethnicity_hhi'],
    student_24['At Risk Count_bbm_smoothed'],
    student_24['student_ethnicity_euclidean_distance'],
   
          

    c=groups.cat.codes,
    cmap='viridis',   
    alpha=0.9,        # slight transparency so overlapping points don't hide each other
    s=20              # smaller points, easier to see the shape of the cloud
)

ax.set_zlabel('Euclidean Distance', labelpad=10)
ax.set_ylabel('At Risk (smoothed)', labelpad=10)
ax.set_xlabel('Student HHI', labelpad=10)
ax.set_title('District clustering view')

fig.colorbar(scatter, ax=ax, label='District size group (0=smallest)')

plt.tight_layout()
plt.show()

In [ ]:
groups = pd.qcut(student_24['All Students Count'], q=6)

# Pull out the three axes and normalize each to 0-1 range
# so the shape isn't stretched weirdly in 3D space
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

x = normalize(student_24['student_ethnicity_euclidean_distance']).values
y = normalize(student_24['student_ethnicity_hhi']).values
z = normalize(student_24['At Risk Count_bbm_smoothed']).values


# Map group codes to colors (viridis-ish: 6 groups, 6 colors)
# OBJ format supports vertex colors in some viewers
color_map = {
    0: (0.267, 0.005, 0.329),  # dark purple  (smallest districts)
    1: (0.283, 0.341, 0.620),  # blue
    2: (0.163, 0.576, 0.558),  # teal
    3: (0.134, 0.738, 0.450),  # green
    4: (0.578, 0.858, 0.140),  # yellow-green
    5: (0.993, 0.906, 0.144),  # yellow        (largest districts)
}

codes = groups.cat.codes.values

with open('districts_3d.obj', 'w') as f:
    f.write("# District clustering point cloud\n")
    f.write(f"# Points: {len(x)}\n")
    f.write("# Axes: X=Euclidean Distance, z=At Risk smoothed, y=Student HHI\n")
    f.write("# Color = district size group (dark=small, bright=large)\n\n")
    
    for i in range(len(x)):
        r, g, b = color_map[codes[i]]
        # 'v x y z r g b' is extended OBJ vertex format with color
        f.write(f"v {x[i]:.6f} {y[i]:.6f} {z[i]:.6f} {r:.3f} {g:.3f} {b:.3f}\n")

In [ ]:
groups = pd.qcut(student_24['All Students Count'], q=6)

# Define the views you want to see
views = [
    (30, 45,  'Default view'),
    (30, 135, 'Rotated 90°'),
    (30, 225, 'Opposite side'),
    (30, 315, 'Last quarter'),
    (85, 45,  'Nearly top-down'),
    (5,  45,  'Nearly ground level'),
]

fig, axes = plt.subplots(
    2, 3,
    figsize=(15, 9),
    subplot_kw={'projection': '3d'}
)

for ax, (elev, azim, title) in zip(axes.flatten(), views):
    sc = ax.scatter(
   
        student_24['student_ethnicity_hhi'],
             student_24['student_ethnicity_euclidean_distance'],
                     student_24['At Risk Count_bbm_smoothed'],

        
       c=groups.cat.codes,
       cmap='viridis',
       alpha=0.6,
        s=15
    )
    ax.view_init(elev=elev, azim=azim)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('HHI', fontsize=7, labelpad=5)
    ax.set_ylabel('Euc. Dist.', fontsize=7, labelpad=5)
    ax.set_zlabel('At Risk', fontsize=7, labelpad=5)
    ax.tick_params(labelsize=6)

fig.colorbar(sc, ax=axes.flatten()[-1], label='District size (0=smallest)')
plt.suptitle('District clustering — 6 orientations', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import itertools

groups = pd.qcut(student_24['All Students Count'], q=6)

# Define your variables — add or remove from here as needed
variables = {
    'Euclidean Dist':  student_24['student_ethnicity_euclidean_distance'],
    'At Risk':         student_24['At Risk Count_bbm_smoothed'],
    'Student HHI':     student_24['student_ethnicity_hhi'],
}

# Get every unique combination of 3 variables assigned to X, Y, Z
# itertools.permutations gives you all orderings so you see every axis assignment
combos = list(itertools.permutations(variables.keys()))

fig, axes = plt.subplots(
    2, 3,
    figsize=(16, 10),
    subplot_kw={'projection': '3d'}
)

for ax, (xvar, yvar, zvar) in zip(axes.flatten(), combos):
    sc = ax.scatter(
        variables[xvar],
        variables[yvar],
        variables[zvar],
        c=groups.cat.codes,
        cmap='viridis',
        alpha=0.7,
        s=15
    )
    ax.view_init(elev=30, azim=45)
    ax.set_xlabel(xvar, fontsize=7, labelpad=5)
    ax.set_ylabel(yvar, fontsize=7, labelpad=5)
    ax.set_zlabel(zvar, fontsize=7, labelpad=5)
    ax.tick_params(labelsize=5)

fig.colorbar(sc, ax=axes.flatten()[-1], label='District size (0=smallest)')
plt.suptitle('All axis permutations — same data, every variable orientation', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# TODO: Figure out nan to 0 issue
# smoothing teacher comp

In [ ]:
# Select Teacher Numerators
#plt.scatter(staff_df_2['Teacher Total Full Time Equiv Count'],staff_df_2['Teacher MS Degree Full Time Equiv Count'] )

Thought: Okay, I actually just want you to clarify my thoughts into a single paragraph so I can remember it and read it. So don't, just write out a paragraph of what I'm saying, clearing it up. So we take the Texas mean, or median, or average, we'll figure it out, but it's one of those numbers that we're trying to find. And I actually just want it to be the mean, I think, if that's the average, because whatever the highest point of its standard distribution score would be. And I want to use that because that's equivalent to what's most standard across all of Texas. And then we'll compute the Euclidean distance from that perfect score, or from that perfect Texas equivalent. And then, and the reason I wanna do that is because when I, I'll now have a better zero to work with, as I'll call it. Or yeah, just a location to reference from how far you are. And maybe, and then I'm thinking, with that, now I can compute the furthest ones from certain distances. And now if I wanted to filter by different things, I could. And I'm thinking it would, yeah.

Chat Output: We define a baseline “reference distribution” for Texas by taking the mean proportion of each ethnicity across all schools (i.e., the average share of each group statewide), treating that as the most typical or standard composition. Each school’s ethnic distribution—optionally smoothed first for stability—is then represented as a vector of proportions and compared to this Texas baseline using Euclidean distance. This distance measures how far a school deviates from the state’s typical makeup, giving us a clear reference point (a “zero”) where a distance of 0 means the school mirrors Texas exactly, and larger values indicate increasing divergence. With this setup, we can rank schools by how representative they are, identify outliers that differ the most from the state norm, and flexibly filter or analyze schools based on how far they sit from that baseline.

## Clustering Algorithms

DBSCAN - (Density-Based Spatial Clustering of Applications with Noise) 

In [ ]:
# Clean each ethnic group's raw proportions using beta-binomial smoothing to stabilize small sample sizes

# Compute the mean of each ethnic group's smoothed proportions across all Texas districts

# Store those means as a single vector this is our "Texas reference distribution" (the perfectly typical Texas school)

# Compute the Euclidean distance between each district's smoothed proportion vector and the Texas reference vector

# Identify the district with the largest Euclidean distance from the reference vector this becomes our d_max

# Normalize each district's distance by d_max, then subtract from 1 to produce a 0–1 representativeness score
